### Device (GPU)

In [50]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


### import important libraries

In [51]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

### Data Transform

In [52]:
transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((48, 48)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

### Load Dataset

In [53]:
train_data = datasets.ImageFolder("../data/combined_data/train", transform=transform)
test_data = datasets.ImageFolder("../data/combined_data/test", transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)



In [54]:
print(train_data.classes)

['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']


### MODEL (RESNET18)

In [55]:

model = models.resnet18(pretrained=True)

# Change input (grayscale)
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

# Output = 7 emotions
model.fc = nn.Linear(model.fc.in_features, 7)

model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\sathi/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


d:\ML_Project\Multimodal Emotion Detection System using Computer Vision and NLP\venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\ML_Project\Multimodal Emotion Detection System using Computer Vision and NLP\venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 44.7M/44.7M [00:04<00:00, 9.56MB/s]


### Loss & Optimizer

In [56]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


### Training Loop

In [57]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}")


Epoch 1, Loss: 962.7777
Epoch 2, Loss: 809.2174
Epoch 3, Loss: 762.8558
Epoch 4, Loss: 699.8034
Epoch 5, Loss: 718.3866
Epoch 6, Loss: 685.1919
Epoch 7, Loss: 629.6641
Epoch 8, Loss: 623.1211
Epoch 9, Loss: 597.9285
Epoch 10, Loss: 561.4681


### Evaluation

In [58]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")


Test Accuracy: 66.41%



### Save Model

In [ ]:
torch.save(model.state_dict(), "../models/face_emotion_model.pth")